# Dataset Exploration

Let's understand the FineWeb data we're training on.

**Key questions:**
- What does the tokenized data look like?
- How are tokens distributed? Are some much more common?
- What's the actual text? Is there a quality signal?
- How does the validation set compare to training?

In [ ]:
import sys
sys.path.insert(0, '../parameter-golf')

import numpy as np
import sentencepiece as spm
from pathlib import Path
from train_gpt_mlx import load_data_shard

# Load tokenizer
sp = spm.SentencePieceProcessor(model_file='../parameter-golf/data/tokenizers/fineweb_1024_bpe.model')
print(f"Tokenizer vocab size: {sp.vocab_size()}")
print(f"Tokenizer type: SentencePiece BPE")

## 1. Vocabulary Analysis

With only 1024 tokens, what does the vocabulary look like?

In [ ]:
# Examine the vocabulary
print("=== First 50 tokens ===")
for i in range(50):
    piece = sp.id_to_piece(i)
    is_byte = sp.is_byte(i)
    is_control = sp.is_control(i)
    is_unk = sp.is_unknown(i)
    flags = []
    if is_byte: flags.append('BYTE')
    if is_control: flags.append('CTRL')
    if is_unk: flags.append('UNK')
    flag_str = f" [{','.join(flags)}]" if flags else ""
    print(f"  {i:4d}: '{piece}'{flag_str}")

print(f"\n=== Last 20 tokens ===")
for i in range(sp.vocab_size() - 20, sp.vocab_size()):
    print(f"  {i:4d}: '{sp.id_to_piece(i)}'")

# Count token types
n_byte = sum(1 for i in range(sp.vocab_size()) if sp.is_byte(i))
n_control = sum(1 for i in range(sp.vocab_size()) if sp.is_control(i))
n_unk = sum(1 for i in range(sp.vocab_size()) if sp.is_unknown(i))
n_regular = sp.vocab_size() - n_byte - n_control - n_unk

print(f"\n=== Token Types ===")
print(f"Regular BPE tokens: {n_regular}")
print(f"Byte tokens:        {n_byte}")
print(f"Control tokens:     {n_control}")
print(f"Unknown tokens:     {n_unk}")

## 2. Training Data: Token Distribution

In [ ]:
# Load first training shard
train_shard = load_data_shard(Path('../parameter-golf/data/datasets/fineweb10B_sp1024/fineweb_train_000000.bin'))
print(f"Shard 0 tokens: {len(train_shard):,}")
print(f"Token dtype: {train_shard.dtype}")
print(f"Token range: [{train_shard.min()}, {train_shard.max()}]")

# Token frequency
counts = np.bincount(train_shard, minlength=1024)
total = counts.sum()

print(f"\n=== Top 20 Most Common Tokens ===")
top_ids = np.argsort(-counts)[:20]
for rank, tid in enumerate(top_ids):
    piece = sp.id_to_piece(int(tid))
    pct = counts[tid] / total * 100
    print(f"  {rank+1:3d}. token {tid:4d} '{piece:10s}' — {counts[tid]:>10,} occurrences ({pct:.2f}%)")

print(f"\n=== Token Usage ===")
used = np.sum(counts > 0)
print(f"Tokens with >0 occurrences: {used}/{len(counts)}")
print(f"Top 10 tokens cover: {counts[top_ids[:10]].sum() / total * 100:.1f}% of all tokens")
print(f"Top 100 tokens cover: {counts[np.argsort(-counts)[:100]].sum() / total * 100:.1f}% of all tokens")

## 3. Decode Some Text

Let's see what the actual training data looks like.

In [ ]:
# Decode first few sequences
for start in [0, 10000, 50000, 100000]:
    chunk = train_shard[start:start+256]
    text = sp.decode(chunk.tolist())
    print(f"\n=== Tokens {start}-{start+256} ===")
    print(text[:500])
    print("...")

## 4. Validation Data Comparison

In [ ]:
# Load validation shard
val_shard = load_data_shard(Path('../parameter-golf/data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin'))
print(f"Validation tokens: {len(val_shard):,}")
print(f"Training shard 0:  {len(train_shard):,}")
print(f"Ratio: val is {len(val_shard)/len(train_shard)*100:.1f}% the size of one train shard")

val_counts = np.bincount(val_shard, minlength=1024)

# Compare distributions
train_freq = counts / counts.sum()
val_freq = val_counts / val_counts.sum()

# KL divergence (rough measure of distribution shift)
# Add smoothing to avoid log(0)
eps = 1e-10
kl = np.sum(val_freq * np.log((val_freq + eps) / (train_freq + eps)))
print(f"\nKL divergence (val || train): {kl:.6f}")
print(f"(Lower = more similar distributions)")

# Show tokens with biggest frequency differences
freq_diff = np.abs(val_freq - train_freq)
diff_ids = np.argsort(-freq_diff)[:10]
print(f"\n=== Tokens with Biggest Train/Val Frequency Difference ===")
for tid in diff_ids:
    print(f"  token {tid:4d} '{sp.id_to_piece(int(tid)):10s}' — train: {train_freq[tid]*100:.3f}%, val: {val_freq[tid]*100:.3f}%")

## 5. Bytes-per-Token Analysis

BPB (bits-per-byte) depends on how many UTF-8 bytes each token represents.
Tokens that represent more bytes give you more "compression credit".

In [ ]:
from train_gpt_mlx import build_sentencepiece_luts

base_bytes_lut, has_leading_space_lut, is_boundary_token_lut = build_sentencepiece_luts(sp, 1024)

# Analyze bytes per token
print("=== Bytes per Token Distribution ===")
for bpt in sorted(set(base_bytes_lut)):
    if bpt == 0: continue
    n_tokens = np.sum(base_bytes_lut == bpt)
    # Weighted by frequency in training data
    mask = base_bytes_lut[np.arange(1024)] == bpt
    freq = np.sum(train_freq[mask])
    print(f"  {bpt} bytes/token: {n_tokens:4d} tokens, {freq*100:.1f}% of training data")

# Average bytes per token
avg_bytes = np.sum(base_bytes_lut[:1024].astype(np.float64) * train_freq)
print(f"\nWeighted average bytes/token: {avg_bytes:.3f}")
print(f"\nThis means on average, each token ≈ {avg_bytes:.1f} bytes of text.")
print(f"BPB formula: bpb = (loss / ln(2)) × (tokens / bytes)")
print(f"So: bpb ≈ loss × {1 / (avg_bytes * np.log(2)):.4f}")

## 6. Shard-to-Shard Variation

Are some shards "harder" or "easier"? This matters for training dynamics.

In [ ]:
data_dir = Path('../parameter-golf/data/datasets/fineweb10B_sp1024')
train_files = sorted(data_dir.glob('fineweb_train_*.bin'))

print(f"Available training shards: {len(train_files)}")
print(f"\n=== Per-Shard Statistics ===")
shard_entropies = []
for f in train_files:
    tokens = load_data_shard(f)
    counts = np.bincount(tokens, minlength=1024)
    freq = counts / counts.sum()
    entropy = -np.sum(freq[freq > 0] * np.log2(freq[freq > 0]))
    shard_entropies.append(entropy)
    unique = np.sum(counts > 0)
    print(f"  {f.name}: {len(tokens):>12,} tokens, {unique} unique tokens, entropy: {entropy:.4f} bits")

if len(shard_entropies) > 1:
    print(f"\nEntropy range: {min(shard_entropies):.4f} - {max(shard_entropies):.4f}")
    print(f"Higher entropy = more diverse text = potentially harder to model")

## Key Takeaways

1. **1024 vocab is tiny** — most tokens are single characters or very short subwords
2. **Token distribution is very skewed** — a few tokens dominate (common letters, spaces)
3. **~3-4 bytes per token on average** — this is the conversion factor for BPB
4. **Val/train distributions should be similar** — if not, there could be a domain shift issue
5. **Shard variation** — some shards may be harder than others

### Data-related opportunities within the rules:
- **Custom tokenizer** — the rules allow this! A better tokenizer could improve compression
- **Data ordering** — order shards by difficulty? Curriculum learning?
- **Sequence length** — longer sequences (2048, 4096) give more context → better predictions